In [ ]:
# opt_test_functions.py
import numpy as np

class BaseFunction:
    def __call__(self, x):
        raise NotImplementedError

class Sphere(BaseFunction):
    def __call__(self, x):
        shifted = x - 0.5
        raw = np.sum(shifted**2, axis=0)
        max_val = 0.25 * x.shape[0]
        return raw / max_val

class Rastrigin(BaseFunction):
    def __call__(self, x):
        d = x.shape[0]
        shifted = x - 0.5
        raw = 10 * d + np.sum(100 * shifted**2 - 10 * np.cos(2 * np.pi * x), axis=0)
        max_val = 10 * d + 100 * 0.25 * d + 10 * d
        return raw / max_val

class Ackley(BaseFunction):
    def __call__(self, x):
        a = 20
        b = 0.2
        c = 2 * np.pi
        d = x.shape[0]
        shifted = x - 0.5
        sum_sq = np.sum(shifted**2, axis=0)
        cos_sum = np.sum(np.cos(c * shifted), axis=0)
        term1 = -a * np.exp(-b * np.sqrt(sum_sq / d))
        term2 = -np.exp(cos_sum / d)
        raw = term1 + term2 + a + np.exp(1)
        max_val = 2 * a + np.exp(1) - 1
        return raw / max_val

class Michalewicz(BaseFunction):
    def __init__(self, m=10):
        self.m = m

    def __call__(self, x):
        d = x.shape[0]
        i = np.arange(1, d + 1).reshape(-1, 1, 1)
        raw = -np.sum(np.sin(x * np.pi) * np.sin(i * x**2 * np.pi)**(2 * self.m), axis=0)
        min_val = -d  # Best value is near -d
        return raw / min_val  # normalize to [0,1]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

myTestFunction = Ackley()

linsp = np.linspace(0, 1, 100)
x1, x2 = np.meshgrid(linsp, linsp)
x = np.stack([x1, x2], axis=0)  # shape (2, 100, 100)
y = myTestFunction(x)  # shape (100, 100)

fig = plt.figure(figsize=(15, 10))
ax = plt.axes(projection='3d')
ax.contour3D(x[0], x[1], y, 150, cmap='gnuplot')
ax.set_zlabel('Response', fontsize=15)
ax.set_title("Ackley Function", fontsize=24)
ax.set_xticklabels([])
ax.set_yticklabels([])
plt.show()


In [ ]:
import numpy as np

def random_search(func_obj, dim, n_iter):
    best_score = float('inf')
    best_x = None
    history = []

    for _ in range(n_iter):
        x = np.random.uniform(0, 1, dim)
        x_input = x.reshape(dim, 1, 1)  # match expected shape: (dim, 1, 1)
        score = func_obj(x_input).item()  # get scalar from array
        history.append(score)
        if score < best_score:
            best_score = score
            best_x = x

    return best_x, best_score, history


In [ ]:
import sys
sys.path.append("../Bayesmark")  # Adjust path to import pfns4bo


import pfns4bo

# copied from `pfns4bo/config.json`
config = {
            "pfn_file": pfns4bo.hebo_plus_model,
            # alternatively give a relative path from pfns4bo
            #"pfn_file" : "final_models/model_hebo_morebudget_9_unused_features_3.pt",
            "minimize": 1,
            "fit_encoder_from_step": None,
            "sample_only_valid": 1,
            "pre_sample_size": 1000,
            "num_candidates": 10,
            "max_initial_design": 1,
            "fixed_initial_guess": 0.0
        }

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
np.warnings = warnings

from pfns4bo.pfn_bo_bayesmark import PFNOptimizer
from bayesmark.experiment import _build_test_problem, run_study, OBJECTIVE_NAMES
import os


#function_instance = _build_test_problem(model_name='ada', dataset='breast', scorer='nll', path=None)
function_instance = _build_test_problem(model_name='ada', dataset='boston', scorer='mse', path=None)

# Setup optimizer
api_config = function_instance.get_api_config()
# check is file

opt = PFNOptimizer(api_config, verbose=True, device="cpu:0", **config)

function_evals, timing, suggest_log = run_study(
    opt, function_instance, n_calls=3, n_suggestions=1, callback=None, n_obj=len(OBJECTIVE_NAMES),
)

In [ ]:

import matplotlib.pyplot as plt

dim = 10
n_iter = 1000

rastrigin = Ackley()
best_x, best_score, history = random_search(rastrigin, dim, n_iter)

print("Best score (normalized):", best_score)


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history)
plt.title("Random Search Convergence")
plt.xlabel("Iteration")
plt.ylabel("Function Value")
plt.grid()
plt.show()
